# Part 1: setup, assignment checklist and correctness checks
Run this notebook first. Then run `model 1/model_1.ipynb` through `model 5/model_5.ipynb`, and finish with `06_comparison.ipynb`. All model code, measurements, plots and result tables are in notebooks. Data, tokenizer and binary checkpoints live beside them.

The five required variants are dense MLP, top-1/4, top-2/4, always-active shared plus top-1/3 routed, and top-2/4 with baseline-matched active parameters. Report target-only test perplexity, corpus BLEU for each source language, and expert-selection heatmaps. Training uses the same realized input-token budget in all five final runs. Hyperparameters are selected on validation data only.

`PART1_PROFILE=full` runs two 1M-token learning-rate pilots and a 60M-token final run for each model. The realized whole-batch budget is printed. `PART1_PROFILE=smoke` is a labeled pipeline check, never a final translation result. The default is `inspect`, which performs checks without launching long training; the model notebooks already contain their completed outputs. On ADA, use the scheduler cells in `07_execution_and_upload.ipynb`.

The assignment also requires checkpoint links on Hugging Face and a repository submission with uv metadata. These remain separately checkable submission items. The PDF prohibits LLM assistance with the assessed writeup; write your own interpretation of the plots and observations.

Each model folder now contains one `model_N.ipynb` with code and completed results, plus `full/checkpoint.pt`. Smoke-run artifacts and duplicate notebooks have been removed. Open the notebooks to view their saved outputs; retraining is unnecessary.


In [1]:
from pathlib import Path
import json, os
here=Path.cwd().resolve()
ROOT=next((p for p in [here,*here.parents,here/'part 1'] if (p/'shared'/'implementation.ipynb').exists()),None)
assert ROOT is not None, 'Set the notebook working directory inside part 1.'
for cell in json.loads((ROOT/'shared'/'implementation.ipynb').read_text())['cells']:
    if cell['cell_type']=='code' and 'definitions' in cell.get('metadata',{}).get('tags',[]):
        exec(compile(cell['source'],'shared/implementation.ipynb','exec'),globals())


In [2]:
import importlib.metadata as metadata
print({'python':platform.python_version(),'device':str(DEVICE),'gpu':torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else None})
display(pd.DataFrame([{'package':p,'version':metadata.version(p)} for p in ('torch','numpy','tokenizers','sacrebleu','pandas','matplotlib','nbformat')]))
counts=pd.DataFrame([parameter_counts(Config(variant=i)) for i in range(1,6)])
assert counts.total.iloc[:4].nunique()==1
assert counts.active_min.iloc[4]==counts.active_max.iloc[4]==counts.total.iloc[0]
display(counts)


{'python': '3.12.4', 'device': 'cuda', 'gpu': 'NVIDIA GeForce RTX 2080 Ti'}


,package,version
0,torch,2.6.0+cu124
1,numpy,2.5.3
2,tokenizers,0.23.2
3,sacrebleu,2.6.0
4,pandas,3.0.5
5,matplotlib,3.11.2
6,nbformat,5.11.1


,model,total,active_min,active_max,expert_widths
0,1,27079680,27079680,27079680,[2048]
1,2,27079680,17648640,17654784,"[512, 512, 511, 511]"
2,3,27079680,20788224,20800512,"[512, 512, 511, 511]"
3,4,27079680,20794368,20800512,"[512, 512, 511, 511]"
4,5,39650304,27079680,27079680,"[1023, 1023, 1023, 1023]"


In [3]:
checks=[run_checks(i) for i in range(1,6)]
display(pd.DataFrame(checks))


,model,causal_attention,padding,target_loss,KV_cache,checkpoint,tiny_overfit_initial,tiny_overfit_final,sparse_dispatch,router_gradient
0,1,PASS,PASS,PASS,PASS,PASS,4.509934,0.094532,not applicable,not applicable
1,2,PASS,PASS,PASS,PASS,PASS,4.460786,0.119522,PASS,PASS
2,3,PASS,PASS,PASS,PASS,PASS,4.451444,0.131763,PASS,PASS
3,4,PASS,PASS,PASS,PASS,PASS,4.448055,0.129771,PASS,PASS
4,5,PASS,PASS,PASS,PASS,PASS,4.468045,0.110933,PASS,PASS


In [4]:
if os.environ.get('PART1_PREPARE','0')=='1':
    manifest=prepare_data()
    display(pd.DataFrame(manifest['audit']).T)
    display(pd.DataFrame(manifest['tokenizer_stats']).T)
elif (ROOT/'shared'/'data'/'manifest.json').exists():
    manifest=json.loads((ROOT/'shared'/'data'/'manifest.json').read_text())
    display(pd.DataFrame(manifest['audit']).T)
else:
    print('Data preparation has not run. Set PART1_PREPARE=1 and rerun this cell.')


/home2/divyansh.atri/anlp_a2/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


train {'original_rows': 446252, 'rows_after_overlap_dedup': 446252, 'removed_target_overlap': 0, 'removed_train_triplet_duplicate': 0, 'retained_triplets': 446202, 'pairs': 892404, 'input_tokens': 38449379, 'target_tokens': 18737614, 'drops': {'overlength_vi': 48, 'triplets_removed': 50, 'overlength_ja': 16}}


validation {'original_rows': 24792, 'rows_after_overlap_dedup': 24792, 'removed_target_overlap': 0, 'removed_train_triplet_duplicate': 0, 'retained_triplets': 24789, 'pairs': 49578, 'input_tokens': 2140582, 'target_tokens': 1043324, 'drops': {'overlength_vi': 3, 'triplets_removed': 3, 'overlength_ja': 1}}


test {'original_rows': 24792, 'rows_after_overlap_dedup': 24792, 'removed_target_overlap': 0, 'removed_train_triplet_duplicate': 0, 'retained_triplets': 24791, 'pairs': 49582, 'input_tokens': 2133061, 'target_tokens': 1038360, 'drops': {'overlength_vi': 1, 'triplets_removed': 1}}


,original_rows,rows_after_overlap_dedup,removed_target_overlap,removed_train_triplet_duplicate,retained_triplets,pairs,input_tokens,target_tokens,drops
train,446252,446252,0,0,446202,892404,38449379,18737614,"{'overlength_vi': 48, 'triplets_removed': 50, ..."
validation,24792,24792,0,0,24789,49578,2140582,1043324,"{'overlength_vi': 3, 'triplets_removed': 3, 'o..."
test,24792,24792,0,0,24791,49582,2133061,1038360,"{'overlength_vi': 1, 'triplets_removed': 1}"


,mean_tokens,roundtrip_checked
vi,22.354,1000.0
ja,20.039,1000.0
en,19.928,1000.0
